# Overview 1

`results/compare/**/*.json` 결과 확인


In [6]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

ROOT = Path("/home/hyeseojeon/data/graph")
COMPARE_DIR = ROOT / "results" / "compare" / "0507" / "4.1-mini"
COMPARE_DIR

PosixPath('/home/hyeseojeon/data/graph/results/compare/0507/4.1-mini')

In [7]:
result_files = sorted(COMPARE_DIR.rglob("*.json"))

print(f"Found {len(result_files)} compare result file(s).")
for path in result_files:
    print(path.relative_to(ROOT))

Found 3 compare result file(s).
results/compare/0507/4.1-mini/2wiki_compare_all.json
results/compare/0507/4.1-mini/hotpotqa_compare_all.json
results/compare/0507/4.1-mini/musique_compare_all.json


In [8]:
def load_records(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("results", "data", "samples"):
            if isinstance(data.get(key), list):
                return data[key]
    raise ValueError(f"Unsupported compare result structure: {path}")

def dataset_name(path: Path) -> str:
    name = path.stem
    for suffix in ("_compare_all", "_vanilla_searchr1", "_vanilla_open-book+gold_searchr1"):
        if suffix in name:
            name = name.split(suffix, 1)[0]
    return name

all_results = []
results_by_dataset = {}

for path in result_files:
    records = load_records(path)
    tagged = []
    for row in records:
        row = dict(row)
        row["_dataset"] = dataset_name(path)
        tagged.append(row)
    results_by_dataset[dataset_name(path)] = tagged
    all_results.extend(tagged)

print(f"Loaded {len(all_results)} sample result(s).")

Loaded 2000 sample result(s).


In [9]:
def label_of(value):
    if isinstance(value, dict):
        return value.get("label")
    return None

sample_rows = []
turn_rows = []

for sample_pos, sample in enumerate(all_results):
    dataset = sample.get("_dataset", "")
    sid = sample.get("uid", sample.get("index", sample_pos))
    turns = sample.get("turn_verifications") or []
    trace = sample.get("searchr1_trace") or {}
    total_results = trace.get("total_search_results") or {}
    docs = total_results.get("document_text") or []
    is_gold = total_results.get("is_gold") or []

    sample_rows.append({
        "dataset": dataset,
        "index": sample.get("index"),
        "uid": sample.get("uid"),
        "question": sample.get("question"),
        "gold_answers": sample.get("gold_answers"),
        "searchr1_answer": sample.get("searchr1_answer"),
        "answer_correct": label_of(sample.get("answer_correct")),
        "num_turns": sample.get("num_turns"),
        "actual_turn_rows": len(turns),
        "num_total_docs": len(docs),
        "num_gold_docs": sum(int(x) for x in is_gold if x is not None),
        "has_prompt": bool(trace.get("prompt")),
        "num_reasoning_steps": len(trace.get("reasoning_steps") or []),
    })

    for turn in turns:
        turn_rows.append({
            "dataset": dataset,
            "sample_index": sample.get("index"),
            "uid": sid,
            "turn": turn.get("turn"),
            "subquery": turn.get("subquery"),
            "question_graph_vs_document": label_of(turn.get("question_graph_vs_document")),
            "question_graph_vs_think": label_of(turn.get("question_graph_vs_think")),
            "think_vs_query": label_of(turn.get("think_vs_query")),
            "document_vs_next_think": label_of(turn.get("document_vs_next_think")),
            "num_docs_in_turn": len(turn.get("document_text") or []),
            "has_next_think": bool(turn.get("next_think")),
        })

sample_df = pd.DataFrame(sample_rows)
turn_df = pd.DataFrame(turn_rows)

print(f"sample_df: {sample_df.shape}")
print(f"turn_df: {turn_df.shape}")

sample_df: (2000, 13)
turn_df: (6486, 11)


## File-Level Summary

In [10]:
if sample_df.empty:
    print("No compare results loaded.")
else:
    file_summary = sample_df.groupby("dataset", dropna=False).agg(
        samples=("uid", "count"),
        turns=("actual_turn_rows", "sum"),
        avg_turns=("actual_turn_rows", "mean"),
        avg_total_docs=("num_total_docs", "mean"),
        total_gold_docs=("num_gold_docs", "sum"),
        samples_with_prompt=("has_prompt", "sum"),
    ).reset_index()
    display(file_summary)

,dataset,samples,turns,avg_turns,avg_total_docs,total_gold_docs,samples_with_prompt
0,2wiki,500,1736,3.472,7.108,869,0
1,hotpotqa,500,1294,2.588,5.952,1090,0
2,musique,1000,3456,3.456,8.196,1174,0


## Answer Correct Label Counts

In [11]:
if not sample_df.empty:
    display(pd.crosstab(sample_df["dataset"], sample_df["answer_correct"], margins=True, dropna=False))

answer_correct,false,true,All
dataset,,,
2wiki,285,215,500
hotpotqa,200,300,500
musique,833,167,1000
All,1318,682,2000


## Turn-Level Label Counts

In [12]:
turn_checks = [
    "question_graph_vs_document",
    "question_graph_vs_think",
    "think_vs_query",
    "document_vs_next_think",
]

if turn_df.empty:
    print("No turn verification rows loaded.")
else:
    for check in turn_checks:
        print(f"\n{check}")
        display(pd.crosstab(turn_df["dataset"], turn_df[check], margins=True, dropna=False))


question_graph_vs_document


question_graph_vs_document,false,true,All
dataset,,,
2wiki,1344,392,1736
hotpotqa,697,597,1294
musique,2237,1219,3456
All,4278,2208,6486



question_graph_vs_think


question_graph_vs_think,false,true,All
dataset,,,
2wiki,100,1636,1736
hotpotqa,116,1178,1294
musique,328,3128,3456
All,544,5942,6486



think_vs_query


think_vs_query,false,true,All
dataset,,,
2wiki,150,1586,1736
hotpotqa,149,1145,1294
musique,400,3056,3456
All,699,5787,6486



document_vs_next_think


document_vs_next_think,fail,false,hallu,true,NaN,All
dataset,,,,,,
2wiki,276,438,49,435,538,1736
hotpotqa,150,206,12,421,505,1294
musique,425,900,60,1064,1007,3456
All,851,1544,121,1920,0,6486


# Overview 2

doc vs. graph / think vs. graph 결과 통합

In [7]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

In [8]:
## think vs. question graph path
# /home/hyeseojeon/data/graph/results/analysis/0410/openbook/2wiki_vanilla_vs_triplet_think_openbook.json

## doc vs. question graph path
# /home/hyeseojeon/data/graph/results/veri_evidence/0411/hotpotqa_veri_evidence.json

from utils.metrics.answer import compute_f1, metric_max_over_ground_truths

ROOT = Path("/home/hyeseojeon/data/graph/results")
THINK_DIR = ROOT / "analysis" / "0410" / "openbook"
DOC_DIR = ROOT / "veri_evidence" / "0411"

PAIR_ORDER = ["0/0", "1/0", "0/1", "1/1"]


def read_json_records(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("results", "data", "samples"):
            if isinstance(data.get(key), list):
                return data[key]
    raise ValueError(f"Unsupported result structure: {path}")


def overview_dataset_name(path: Path) -> str:
    name = path.stem
    for suffix in (
        "_vanilla_vs_triplet_think_openbook",
        "_vanilla_vs_triplet_openbook",
        "_veri_evidence",
    ):
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name.split("_", 1)[0]


def is_pass(value) -> int:
    if isinstance(value, dict):
        value = value.get("label") or value.get("answer") or value.get("value")
    if isinstance(value, bool):
        return int(value)
    if value is None:
        return 0
    return int(str(value).strip().lower() in {"1", "true", "yes", "y", "pass", "passed", "support", "supported", "correct"})


def qa_correct(row) -> int:
    for key in ("answer_matches_gold", "qa_correct", "answer_correct", "correct", "is_correct"):
        if key in row:
            value = row[key]
            if isinstance(value, dict):
                value = value.get("label")
            return is_pass(value)
    return 0


def gold_answers(row):
    for key in ("gold_answers", "answer_aliases", "answers"):
        value = row.get(key)
        if isinstance(value, list) and value:
            return [str(x) for x in value]
    answer = row.get("answer")
    if isinstance(answer, list):
        return [str(x) for x in answer]
    if answer is None:
        return []
    return [str(answer)]


def qa_f1(row):
    prediction = row.get("predicted_answer") or row.get("searchr1_answer") or row.get("response") or ""
    answers = gold_answers(row)
    if not answers:
        return 0.0
    return float(metric_max_over_ground_truths(compute_f1, str(prediction), answers))


def row_key(row):
    if row.get("uid") is not None:
        return ("uid", row["uid"])
    return ("index", row.get("index"))


def load_by_dataset(files):
    by_dataset = {}
    for path in files:
        dataset = overview_dataset_name(path)
        by_dataset[dataset] = {
            row_key(row): dict(row)
            for row in read_json_records(path)
        }
    return by_dataset


think_files = sorted(THINK_DIR.glob("*_vanilla_vs_triplet_think_openbook.json"))
doc_files = sorted(DOC_DIR.glob("*_veri_evidence.json"))

think_by_dataset = load_by_dataset(think_files)
doc_by_dataset = load_by_dataset(doc_files)

print(f"Found {len(think_files)} think result file(s).")
print(f"Found {len(doc_files)} doc result file(s).")
print("Datasets:", sorted(set(think_by_dataset) & set(doc_by_dataset)))


Found 3 think result file(s).
Found 3 doc result file(s).
Datasets: ['2wiki', 'hotpotqa', 'musique']


In [9]:
overview_tables = {}
overview_rows = []

for dataset in sorted(set(think_by_dataset) & set(doc_by_dataset)):
    think_rows = think_by_dataset[dataset]
    doc_rows = doc_by_dataset[dataset]
    common_keys = sorted(set(think_rows) & set(doc_rows), key=lambda x: (x[0], str(x[1])))

    pair_items = []
    for key in common_keys:
        think_row = think_rows[key]
        doc_row = doc_rows[key]
        doc_pass = is_pass(doc_row.get("document_supported"))
        think_pass = is_pass(think_row.get("align"))
        pair_items.append({
            "dataset": dataset,
            "key_type": key[0],
            "key": key[1],
            "pair": f"{doc_pass}/{think_pass}",
            "qa_correct": qa_correct(think_row),
            "f1_value": qa_f1(think_row),
        })

    pair_df = pd.DataFrame(pair_items)
    if pair_df.empty:
        table = pd.DataFrame({
            "pair": PAIR_ORDER,
            "n": 0,
            "qa_correct": 0,
            "qa_wrong": 0,
            "acc": 0.0,
            "f1": 0.0,
        })
    else:
        table = (
            pair_df.groupby("pair", dropna=False)
            .agg(
                n=("pair", "size"),
                qa_correct=("qa_correct", "sum"),
                f1=("f1_value", "mean"),
            )
            .reindex(PAIR_ORDER, fill_value=0)
            .reset_index()
        )
        table["qa_correct"] = table["qa_correct"].astype(int)
        table["qa_wrong"] = table["n"] - table["qa_correct"]
        table["acc"] = table["qa_correct"] / table["n"].replace(0, pd.NA)
        table["acc"] = table["acc"].fillna(0.0).round(3)
        table["f1"] = table["f1"].fillna(0.0).round(3)
        table = table[["pair", "n", "qa_correct", "qa_wrong", "acc", "f1"]]

    overview_tables[dataset] = table
    overview_rows.append(table.assign(dataset=dataset))

    print(f"\n{dataset}")
    display(table)

overview_summary_df = pd.concat(overview_rows, ignore_index=True) if overview_rows else pd.DataFrame(
    columns=["dataset", "pair", "n", "qa_correct", "qa_wrong", "acc", "f1"]
)



2wiki


,pair,n,qa_correct,qa_wrong,acc,f1
0,0/0,176,46,130,0.261,0.278
1,1/0,13,5,8,0.385,0.513
2,0/1,206,78,128,0.379,0.405
3,1/1,105,64,41,0.610,0.717



hotpotqa


,pair,n,qa_correct,qa_wrong,acc,f1
0,0/0,111,33,78,0.297,0.347
1,1/0,48,28,20,0.583,0.604
2,0/1,126,59,67,0.468,0.532
3,1/1,215,150,65,0.698,0.802



musique


,pair,n,qa_correct,qa_wrong,acc,f1
0,0/0,400,18,382,0.045,0.082
1,1/0,39,8,31,0.205,0.286
2,0/1,370,32,338,0.086,0.155
3,1/1,191,46,145,0.241,0.354


# Overview 3

`results/compare/0513/2wiki_compare_one.json` question-level pair 결과

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from utils.metrics.answer import compute_exact, compute_f1, metric_max_over_ground_truths

PAIR_ORDER = ["0/0", "1/0", "0/1", "1/1"]
COMPARE_ONE_DIR = Path("/home/hyeseojeon/data/graph/results/compare/0513")
COMPARE_ONE_FILES = {
    "2wiki": COMPARE_ONE_DIR / "2wiki_compare_one.json",
    "hotpot": COMPARE_ONE_DIR / "hotpotqa_compare_one.json",
    "musique": COMPARE_ONE_DIR / "musique_compare_one.json",
}


def read_json_records(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("results", "data", "samples"):
            if isinstance(data.get(key), list):
                return data[key]
    raise ValueError(f"Unsupported result structure: {path}")


def gold_answers(row):
    for key in ("gold_answers", "answer_aliases", "answers"):
        value = row.get(key)
        if isinstance(value, list) and value:
            return [str(x) for x in value]
    answer = row.get("answer")
    if isinstance(answer, list):
        return [str(x) for x in answer]
    if answer is None:
        return []
    return [str(answer)]


def is_pass(value) -> int:
    if isinstance(value, dict):
        value = value.get("label") or value.get("answer") or value.get("value")
    if isinstance(value, bool):
        return int(value)
    if value is None:
        return 0
    return int(str(value).strip().lower() in {"1", "true", "yes", "y", "pass", "passed", "support", "supported", "correct"})


def qa_f1(row):
    prediction = row.get("predicted_answer") or row.get("searchr1_answer") or row.get("response") or ""
    answers = gold_answers(row)
    if not answers:
        return 0.0
    return float(metric_max_over_ground_truths(compute_f1, str(prediction), answers))


def qa_correct_with_answer_fallback(row) -> int:
    for key in ("answer_matches_gold", "qa_correct", "answer_correct", "correct", "is_correct"):
        if key not in row:
            continue
        value = row[key]
        if isinstance(value, dict):
            value = value.get("label")
        return is_pass(value)
    prediction = row.get("predicted_answer") or row.get("searchr1_answer") or row.get("response") or ""
    answers = gold_answers(row)
    if not answers:
        return 0
    return int(metric_max_over_ground_truths(compute_exact, str(prediction), answers))

def sample_pair(sample) -> str:
    doc_think_ok = sample.get("doc_think_ok") or {}
    pair = doc_think_ok.get("sample_pair")
    if pair is not None:
        return pair
    doc_pass = is_pass(doc_think_ok.get("doc_ok_all"))
    think_pass = is_pass(doc_think_ok.get("think_ok_all"))
    return f"{doc_pass}/{think_pass}"


def compare_one_pair_rows(dataset: str, path: Path):
    rows = []
    for sample in read_json_records(path):
        sample_correct = qa_correct_with_answer_fallback(sample)
        sample_f1 = qa_f1(sample)
        rows.append({
            "dataset": dataset,
            "key_type": "uid" if sample.get("uid") is not None else "index",
            "key": sample.get("uid", sample.get("index")),
            "pair": sample_pair(sample),
            "qa_correct": sample_correct,
            "f1_value": sample_f1,
        })
    return rows


def overview3_pair_table(pair_df: pd.DataFrame):
    if pair_df.empty:
        return pd.DataFrame({
            "pair": PAIR_ORDER,
            "n": 0,
            "qa_correct": 0,
            "qa_wrong": 0,
            "acc": 0.0,
            "f1": 0.0,
        })

    table = (
        pair_df.groupby("pair", dropna=False)
        .agg(
            n=("pair", "size"),
            qa_correct=("qa_correct", "sum"),
            f1=("f1_value", "mean"),
        )
        .reindex(PAIR_ORDER, fill_value=0)
        .reset_index()
    )
    table["qa_correct"] = table["qa_correct"].astype(int)
    table["qa_wrong"] = table["n"] - table["qa_correct"]
    table["acc"] = table["qa_correct"] / table["n"].replace(0, pd.NA)
    table["acc"] = table["acc"].fillna(0.0).round(3)
    table["f1"] = table["f1"].fillna(0.0).round(3)
    return table[["pair", "n", "qa_correct", "qa_wrong", "acc", "f1"]]


overview3_tables = {}
overview3_rows = []

for dataset, path in COMPARE_ONE_FILES.items():
    compare_one_df = pd.DataFrame(compare_one_pair_rows(dataset, path))
    overview3_table = overview3_pair_table(compare_one_df)
    overview3_tables[dataset] = overview3_table
    overview3_rows.append(overview3_table.assign(dataset=dataset))

    display(Markdown(f"### {dataset}"))
    display(overview3_table)

overview3_summary_df = pd.concat(overview3_rows, ignore_index=True) if overview3_rows else pd.DataFrame(
    columns=["pair", "n", "qa_correct", "qa_wrong", "acc", "f1", "dataset"]
)


# Additions

## Empty / Error Cases

In [ ]:
if not sample_df.empty:
    empty_or_suspicious = sample_df[
        (sample_df["actual_turn_rows"] == 0)
        | (sample_df["num_total_docs"] == 0)
        | (sample_df["num_reasoning_steps"] == 0)
        | (sample_df["answer_correct"] == "error")
    ]
    display(empty_or_suspicious)

if not turn_df.empty:
    error_turns = turn_df[
        turn_df[turn_checks].eq("error").any(axis=1)
    ]
    display(error_turns)

## Sample-Level Results

In [ ]:
display(sample_df)

## Turn-Level Results

In [ ]:
display(turn_df)

## Full JSON Results

아래 셀은 로드된 전체 compare JSON을 그대로 출력합니다. 결과가 많으면 노트북 출력이 커질 수 있습니다.

In [ ]:
display(JSON(results_by_dataset, expanded=False))